# v27 Architecture Schematic — Landscape Publication Version

Horizontal-flow methodology figure. All values from checkpoint.

In [ ]:
import torch, math
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle

mpl.rcParams.update({"font.family": "DejaVu Sans", "mathtext.fontset": "dejavusans",
                      "figure.facecolor": "white", "savefig.facecolor": "white"})

CKPT = Path("../../checkpoints/full_run_cloud_v27/best.ckpt")
FIG_DIR = Path("../../analysis_outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("OK")

In [ ]:
ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
cfg = ckpt["hyper_parameters"]["cfg"]

C = dict(
    d=cfg["d_model"], enc_L=cfg["enc_layers"], dec_L=cfg["dec_layers"],
    heads=cfg["enc_heads"], mask_r=cfg["mask_ratio"],
    patch=cfg["temporal_patch"], W=cfg["window"], N=cfg["num_stations"],
    max_delta=cfg["max_delta"],
    stride=cfg.get("delta_grid_stride", 3) or 3,
    d_ff=int(cfg["d_model"] * cfg["mlp_ratio"]),
)
C["W_p"] = C["W"] // C["patch"]
C["K"] = C["max_delta"] // C["stride"] + 1
C["n_params"] = sum(v.numel() for v in ckpt["state_dict"].values()
                     if hasattr(v, "numel")) / 1e6
for k, v in C.items():
    print(f"  {k:12s} = {v}")

In [ ]:
COL = dict(
    input="#E8A838", embed="#F5F0E0", mask_ok="#E8A838", mask_no="#FFCCCC",
    mask_ec="#C0392B", enc_bg="#E8EDF2", t_attn="#D35F5F", s_attn="#5BA55B",
    ffn="#7BAFD4", dec_bg="#F0E8E0", cross="#D4A76A", query="#7B68AE",
    out="#2C7A5E", loss="#7B68AE", res="#555555", txt="#222222",
    ltxt="#777777", block="#F5F7FA", white="#FFFFFF",
)
print("OK")

In [ ]:
def box(ax, x, y, w, h, label="", fc="#FFF", ec="#333",
       lw=0.8, fs=8, tc="#222", zorder=2, bold=False):
    b = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.008",
                        fc=fc, ec=ec, lw=lw, zorder=zorder)
    ax.add_patch(b)
    if label:
        ax.text(x+w/2, y+h/2, label, ha="center", va="center",
                fontsize=fs, color=tc, fontweight="bold" if bold else "normal",
                zorder=zorder+1)
    return b

def arrow(ax, x0, y0, x1, y1, c="#333", lw=0.9, style="-|>", zorder=3):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle=style, color=c, lw=lw), zorder=zorder)

def skip_line(ax, x0, y0, x1, y1, c="#555", lw=0.5):
    ax.plot([x0, x1], [y0, y0], c=c, lw=lw, ls="--", zorder=2)
    ax.plot([x1, x1], [y0, y1], c=c, lw=lw, ls="--", zorder=2)

def repeat(ax, x, y, n, fs=7.5, c="#555"):
    ax.text(x, y, f"\u00d7{n}", ha="center", va="center", fontsize=fs,
            fontweight="bold", color=c,
            bbox=dict(fc="white", ec=c, boxstyle="round,pad=0.12", lw=0.5),
            zorder=5)

print("OK")

In [ ]:
fig, ax = plt.subplots(figsize=(18, 7.5))
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.axis("off")
ax.set_aspect("auto")

sx = dict(inp=0.02, emb=0.13, msk=0.24, enc=0.40, dec=0.64, pred=0.82, loss=0.93)
MID = 0.50

# ── 1. INPUT GRID ──
gx0, gy0 = sx["inp"], MID + 0.12
gw, gh = 0.007, 0.018
nr, nc = 6, 8
for r in range(nr):
    for c_ in range(nc):
        ax.add_patch(Rectangle((gx0+c_*gw*1.15, gy0-r*gh*1.15),
                     gw, gh, fc=COL["input"], ec="#999", lw=0.25, zorder=3))

ax.text(gx0+nc*gw*1.15/2, gy0+gh+0.025,
        r"$\mathbf{X}$", ha="center", fontsize=11, fontweight="bold",
        color=COL["txt"])
ax.text(gx0+nc*gw*1.15/2, gy0+gh+0.008,
        f"N={C['N']}  W={C['W']}  V=6", ha="center", fontsize=6.5,
        color=COL["ltxt"])
ax.text(gx0-0.008, gy0-nr*gh*1.15/2+gh/2, "stations",
        ha="right", va="center", fontsize=5.5, color=COL["ltxt"], rotation=90)
ax.text(gx0+nc*gw*1.15/2, gy0-nr*gh*1.15-0.008, "time steps",
        ha="center", fontsize=5.5, color=COL["ltxt"])

inp_right = gx0 + nc*gw*1.15 + 0.005
inp_mid = gy0 - nr*gh*1.15/2 + gh/2
arrow(ax, inp_right, inp_mid, sx["emb"]-0.005, inp_mid)

# ── 2. EMBEDDINGS ──
ew, eh = 0.075, 0.42
ey = MID - eh/2
box(ax, sx["emb"], ey, ew, eh, fc=COL["embed"], ec="#AAA")

emb_labels = ["value\n(MLP)", "position\n(Fourier)", "topo\n(MLP)",
              "time\n(Fourier)", "step\n(Fourier)"]
emb_colors = [COL["input"], COL["s_attn"], COL["s_attn"],
              "#4A90C4", COL["query"]]
n_emb = len(emb_labels)
emb_h = 0.055
emb_gap = (eh - n_emb*emb_h) / (n_emb+1)
for i, (lbl, col) in enumerate(zip(emb_labels, emb_colors)):
    ey_i = ey + eh - emb_gap*(i+1) - emb_h*(i+1)
    box(ax, sx["emb"]+0.005, ey_i, ew-0.01, emb_h,
        label=lbl, fc=col, ec="#888", fs=5.5, tc="white", lw=0.4)

ax.text(sx["emb"]+ew/2, ey+eh+0.015, "Embeddings",
        ha="center", fontsize=8, fontweight="bold", color=COL["txt"])
ax.text(sx["emb"]+ew/2, ey-0.012, r"$\Sigma$ \u2192 LayerNorm",
        ha="center", fontsize=6.5, color=COL["ltxt"])
ax.text(sx["emb"]+ew/2, ey-0.028, f"d = {C['d']}", ha="center",
        fontsize=6, color=COL["ltxt"])

arrow(ax, sx["emb"]+ew+0.005, inp_mid, sx["msk"]-0.005, inp_mid)

# ── 3. MAE MASKING ──
mx0, my0 = sx["msk"]+0.005, MID + 0.12
mr, mc = 6, 8
masked = {1, 3, 4}
for r in range(mr):
    for c_ in range(mc):
        if r in masked:
            ax.add_patch(Rectangle((mx0+c_*gw*1.15, my0-r*gh*1.15),
                         gw, gh, fc=COL["mask_no"], ec=COL["mask_ec"],
                         lw=0.3, hatch="////", zorder=3))
        else:
            ax.add_patch(Rectangle((mx0+c_*gw*1.15, my0-r*gh*1.15),
                         gw, gh, fc=COL["mask_ok"], ec="#999",
                         lw=0.25, zorder=3))

ax.text(mx0+mc*gw*1.15/2, my0+gh+0.015, "MAE masking",
        ha="center", fontsize=8, fontweight="bold", color=COL["mask_ec"])
ax.text(mx0+mc*gw*1.15/2, my0+gh+0.002,
        f"r = {C['mask_r']}  (whole-station)",
        ha="center", fontsize=6, color=COL["mask_ec"])

leg_y = my0 - mr*gh*1.15 - 0.015
ax.add_patch(Rectangle((mx0, leg_y), 0.008, 0.008,
             fc=COL["mask_ok"], ec="#999", lw=0.3))
ax.text(mx0+0.012, leg_y+0.004, "visible", fontsize=5.5,
        va="center", color=COL["ltxt"])
ax.add_patch(Rectangle((mx0+0.05, leg_y), 0.008, 0.008,
             fc=COL["mask_no"], ec=COL["mask_ec"], lw=0.3, hatch="////"))
ax.text(mx0+0.062, leg_y+0.004, "masked", fontsize=5.5,
        va="center", color=COL["mask_ec"])

msk_right = mx0 + mc*gw*1.15 + 0.008
pm_x = (msk_right + sx["enc"] - 0.01) / 2
ax.text(pm_x, inp_mid+0.012, "Patch merge",
        ha="center", fontsize=6.5, color=COL["ltxt"], fontstyle="italic")
ax.text(pm_x, inp_mid-0.005,
        f"P={C['patch']}: W={C['W']}\u2192{C['W_p']}",
        ha="center", fontsize=5.5, color=COL["ltxt"])
arrow(ax, msk_right, inp_mid, sx["enc"]-0.005, inp_mid)

# ── 4. ENCODER ──
enc_w, enc_h = 0.17, 0.72
enc_y = MID - enc_h/2
box(ax, sx["enc"], enc_y, enc_w, enc_h, fc=COL["enc_bg"], ec="#8899AA", lw=1.0)
ax.text(sx["enc"]+enc_w/2, enc_y+enc_h-0.015, "Encoder",
        ha="center", fontsize=10, fontweight="bold", color=COL["txt"])

bx = sx["enc"] + 0.012
bw = enc_w - 0.024
blk_y0 = enc_y + enc_h - 0.06
row_h = 0.065
gap = 0.015

# Temporal attention
ty = blk_y0 - row_h
box(ax, bx, ty, bw, row_h, label="Temporal\nself-attention",
    fc=COL["t_attn"], tc="white", fs=7, lw=0.5)
skip_line(ax, bx+bw, ty+row_h/2, bx+bw+0.003, ty-gap+row_h/2)
arrow(ax, bx+bw/2, ty, bx+bw/2, ty-gap+row_h, c=COL["res"], lw=0.6)

# Spatial attention
sy = ty - gap - row_h
box(ax, bx, sy, bw, row_h, label="Spatial\nself-attention",
    fc=COL["s_attn"], tc="white", fs=7, lw=0.5)
skip_line(ax, bx+bw, sy+row_h/2, bx+bw+0.003, sy-gap+row_h/2)
arrow(ax, bx+bw/2, sy, bx+bw/2, sy-gap+row_h, c=COL["res"], lw=0.6)

# FFN
fy = sy - gap - row_h
box(ax, bx, fy, bw, row_h, label="FFN",
    fc=COL["ffn"], tc="white", fs=7, lw=0.5)
skip_line(ax, bx+bw, fy+row_h/2, bx+bw+0.003, fy-gap/2)

repeat(ax, bx+bw/2, fy-0.025, C["enc_L"])
ax.text(bx+bw/2, fy-0.055,
        f"h={C['heads']}  d={C['d']}", ha="center", fontsize=5.5, color=COL["ltxt"])
ax.text(bx+bw/2, fy-0.07,
        f"FFN: {C['d']}\u2192{C['d_ff']}\u2192{C['d']}", ha="center", fontsize=5, color=COL["ltxt"])

ln_y = enc_y + 0.008
box(ax, bx+0.02, ln_y, bw-0.04, 0.025, label="LayerNorm",
    fc="#E0E4E8", fs=6, lw=0.4)

# K,V arrow
enc_right = sx["enc"] + enc_w
kv_y = inp_mid + 0.04
arrow(ax, enc_right+0.005, kv_y, sx["dec"]-0.005, kv_y, c=COL["s_attn"], lw=1.2)
ax.text((enc_right+sx["dec"])/2, kv_y+0.018, "K, V",
        ha="center", fontsize=9, fontweight="bold", color=COL["s_attn"])

# ── 5. DECODER ──
dec_w, dec_h = 0.14, 0.72
dec_y = MID - dec_h/2
box(ax, sx["dec"], dec_y, dec_w, dec_h, fc=COL["dec_bg"], ec="#AA9988", lw=1.0)
ax.text(sx["dec"]+dec_w/2, dec_y+dec_h-0.015, "Decoder",
        ha="center", fontsize=10, fontweight="bold", color=COL["txt"])

dx = sx["dec"] + 0.01
dw = dec_w - 0.02

qy_top = dec_y + dec_h - 0.045
box(ax, dx, qy_top, dw, 0.025, label="Station \u00d7 lead queries",
    fc=COL["query"], tc="white", fs=6, lw=0.4)
ax.text(dx+dw/2, qy_top-0.012,
        f"N\u00d7K = {C['N']}\u00d7{C['K']}", ha="center",
        fontsize=5.5, color=COL["ltxt"])

q_arr_y = qy_top - 0.025
ax.text(sx["dec"]-0.025, q_arr_y+0.01, "Q", fontsize=9,
        fontweight="bold", color=COL["query"], ha="center", va="center")
arrow(ax, sx["dec"]-0.012, q_arr_y+0.01, sx["dec"]+0.01, q_arr_y+0.01,
      c=COL["query"], lw=0.8)

sa_y = qy_top - 0.055
box(ax, dx, sa_y, dw, 0.04, label="Self-attention",
    fc=COL["t_attn"], tc="white", fs=6.5, lw=0.5)
arrow(ax, dx+dw/2, sa_y+0.04, dx+dw/2, sa_y+0.04+0.01, c=COL["res"], lw=0.6)

ca_y = sa_y - 0.06
box(ax, dx, ca_y, dw, 0.04, label="Cross-attention",
    fc=COL["cross"], tc="white", fs=6.5, lw=0.5)
arrow(ax, sx["dec"]-0.005, ca_y+0.02, dx, ca_y+0.02,
      c=COL["s_attn"], lw=0.8)
arrow(ax, dx+dw/2, ca_y+0.04, dx+dw/2, sa_y, c=COL["res"], lw=0.6)

df_y = ca_y - 0.06
box(ax, dx, df_y, dw, 0.04, label="FFN",
    fc=COL["ffn"], tc="white", fs=6.5, lw=0.5)
arrow(ax, dx+dw/2, df_y+0.04, dx+dw/2, ca_y, c=COL["res"], lw=0.6)

repeat(ax, dx+dw/2, df_y-0.025, C["dec_L"])
ax.text(dx+dw/2, df_y-0.052,
        f"h={C['heads']}  d={C['d']}", ha="center", fontsize=5.5, color=COL["ltxt"])

rh_y = dec_y + 0.04
box(ax, dx, rh_y, dw, 0.03, label="+ persistence base",
    fc="#E8F5E9", ec=COL["out"], fs=5.5, lw=0.5)
ax.text(dx+dw/2, rh_y-0.012,
        "visible: last obs / masked: 0",
        ha="center", fontsize=5, color=COL["ltxt"])

arrow(ax, sx["dec"]+dec_w+0.005, inp_mid, sx["pred"]-0.005, inp_mid,
      c=COL["out"], lw=1.2)

# ── 6. PREDICTIONS ──
pred_x = sx["pred"]
fan_w = 0.07
fan_h_total = 0.50
fan_y0 = MID + fan_h_total/2
n_leads = C["K"]
lead_h = fan_h_total / n_leads
lead_gap = 0.003

ax.text(pred_x+fan_w/2, fan_y0+0.04, "Outputs",
        ha="center", fontsize=9, fontweight="bold", color=COL["out"])
n_forecast = n_leads - 1
ax.text(pred_x+fan_w/2, fan_y0+0.02,
        f"1 inpainting + {n_forecast} forecasts",
        ha="center", fontsize=6.5, color=COL["ltxt"])

for k in range(n_leads):
    ly = fan_y0 - k*(lead_h) - lead_gap*k
    lh = lead_h - lead_gap
    delta_h = k * C["stride"] / 60
    alpha = 0.4 + 0.6 * (1 - k/(n_leads-1))
    ax.add_patch(Rectangle((pred_x, ly-lh), fan_w, lh,
                 fc=COL["out"], ec="#1A5E40", lw=0.3, alpha=alpha, zorder=3))
    if k in (0, 1, 3, 6, 12):
        if delta_h == 0:
            label = "\u0394=0 (inpainting)"
        else:
            label = f"\u0394={delta_h:.1f}h"
        ax.text(pred_x+fan_w+0.005, ly-lh/2, label,
                fontsize=5, va="center", color=COL["ltxt"])

for k in range(n_leads):
    ly = fan_y0 - k*(lead_h) - lead_gap*k - lead_h/2
    ax.plot([pred_x-0.008, pred_x], [inp_mid, ly],
            color=COL["out"], lw=0.3, alpha=0.5, zorder=2)

# ── 7. LOSS ──
loss_x = sx["loss"]
loss_y = MID - 0.06
loss_w, loss_h_ = 0.065, 0.12
box(ax, loss_x, loss_y, loss_w, loss_h_, fc="#F3EFF8", ec=COL["loss"], lw=0.8)
ax.text(loss_x+loss_w/2, loss_y+loss_h_-0.015, "Huber",
        ha="center", fontsize=8, fontweight="bold", color=COL["loss"])
ax.text(loss_x+loss_w/2, loss_y+loss_h_-0.035, r"$\delta$=1",
        ha="center", fontsize=7, color=COL["loss"])
ax.text(loss_x+loss_w/2, loss_y+loss_h_/2-0.015,
        "uniform\nover K leads", ha="center", fontsize=5.5, color=COL["ltxt"])

arrow(ax, pred_x+fan_w/2, fan_y0 - n_leads*(lead_h+lead_gap) - 0.005,
      loss_x+loss_w/2, loss_y+loss_h_+0.005, c=COL["loss"])

mask_note_y = loss_y - 0.03
ax.add_patch(Rectangle((loss_x, mask_note_y), 0.008, 0.008,
             fc=COL["mask_no"], ec=COL["mask_ec"], lw=0.3, hatch="////"))
ax.text(loss_x+0.012, mask_note_y+0.004, "encoder mask",
        fontsize=5, va="center", color=COL["mask_ec"])
ax.add_patch(Rectangle((loss_x, mask_note_y-0.015), 0.008, 0.008,
             fc=COL["loss"], ec=COL["loss"], lw=0.3, alpha=0.3))
ax.text(loss_x+0.012, mask_note_y-0.011, "loss: all sensors",
        fontsize=5, va="center", color=COL["loss"])

# ── 8. TITLE ──
ax.text(0.50, 0.97, "Weather Station Transformer (v27)",
        ha="center", fontsize=14, fontweight="bold", color=COL["txt"])
ax.text(0.50, 0.935,
        f"\u2248 {C['n_params']:.1f} M parameters",
        ha="center", fontsize=8, color=COL["ltxt"], fontstyle="italic")
ax.text(0.50, 0.025,
        "Inference: no masking (r = 0), all stations \u2192 encoder \u2192 decoder \u2192 12 forecasts + 1 reconstruction",
        ha="center", fontsize=7, color=COL["ltxt"], fontstyle="italic")

plt.tight_layout(pad=0.3)
print("Figure built")


In [ ]:
fig.savefig(FIG_DIR / "v27_architecture.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "v27_architecture.svg", bbox_inches="tight")
fig.savefig(FIG_DIR / "v27_architecture.png", dpi=600, bbox_inches="tight")
print(f"Saved to {FIG_DIR}/v27_architecture.{{pdf,svg,png}}")
plt.show()

## Proposed thesis caption

**Figure X.** Architecture of the Weather Station Transformer (v27).
Observations from N SwissMetNet stations over a 12 h window (W steps at
10 min resolution, V = 6 variables) are projected to d-dimensional tokens
via five summed embeddings (value, position, topography, time, step index).
At training time, 50 % of stations are masked across the full temporal
window (whole-station MAE masking). Visible tokens are temporally patched
(groups of P) and processed by L encoder blocks, each applying factorised
attention: temporal self-attention within each station, then spatial
self-attention across stations at each time step, followed by a feed-forward
network, with pre-LayerNorm residual connections and stochastic depth.
The decoder constructs N × K lead-conditioned station queries and processes
them with cross-attention blocks where queries attend to encoder
representations. A residual head adds the last visible observation as a
persistence prior (zeroed for masked stations). Training minimises Huber
loss (δ = 1), averaged uniformly over all K forecast horizons,
supervising all stations with available sensors. All values in this figure
are read from the v27 checkpoint.

---
## Auto-generated architecture graph (torchview)

`torchview` traces the actual forward pass and renders the computational graph
via Graphviz. Install if needed: `pip install torchview graphviz`.

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == "analysis":
    os.chdir("../..")
if os.path.join(os.getcwd(), "src") not in sys.path:
    sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import subprocess
for pkg in ['torchview', 'graphviz']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import torch
from torchview import draw_graph
from model.mae import StationMAE
from pathlib import Path

print('torchview + graphviz ready')

In [ ]:
# Build v27 model from checkpoint config
CKPT = Path('checkpoints/full_run_cloud_v27/best.ckpt')
ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)
cfg = ckpt['hyper_parameters']['cfg']

mae = StationMAE.from_cfg(cfg, dropout=0.0)
mae.eval()

W = cfg['window']           # 72
N = cfg['num_stations']     # 155
V = 6
stride = int(cfg.get('delta_grid_stride', 3) or 3)
K = cfg['max_delta'] // stride + 1  # 13
B = 1

print(f'd_model={cfg["d_model"]}, enc={cfg["enc_layers"]}L, '
      f'dec={cfg["dec_layers"]}L, heads={cfg["enc_heads"]}, '
      f'patch={cfg.get("temporal_patch",1)}, K={K}')

In [ ]:
# Wrap forward_multi_delta for torchview tracing
import torch.nn as nn

class MAETrace(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m
    def forward(self, x, x_mask, spatial, x_hours, y, y_mask, y_hours, delta_steps):
        return self.m.forward_multi_delta(x, x_mask, spatial, x_hours,
                                          y, y_mask, y_hours, delta_steps)[1]

grid = torch.arange(0, cfg['max_delta'] + 1, stride)
example = (
    torch.randn(B, W, N, V),         # x
    torch.ones (B, W, N, V),         # x_mask
    torch.randn(N, 15),              # spatial
    torch.rand (B, W) * 1e5,         # x_hours
    torch.randn(B, K, N, V),         # y
    torch.ones (B, K, N, V),         # y_mask
    torch.rand (B, K) * 1e5,         # y_hours
    grid.unsqueeze(0).repeat(B, 1),  # delta_steps (B, K)
)

# depth=3 keeps the graph readable — increase for more detail
graph = draw_graph(
    MAETrace(mae),
    input_data=example,
    depth=3,
    graph_name='Station-MAE v27',
    expand_nested=True,
    graph_dir='LR',          # left-to-right layout
    save_graph=True,
    filename='v27_torchview',
    directory=str(Path('analysis_outputs/figures')),
)

print('Saved torchview graph → analysis_outputs/figures/v27_torchview.png')
graph.visual_graph

The torchview graph above shows every module traced during the forward pass.
It is useful for verifying the actual data flow but too dense for a thesis figure.
The **Graphviz schematic** below provides a cleaner, hand-designed version.

---
## Publication-quality schematic (Graphviz)

A hand-designed horizontal-flow diagram using Graphviz for clean layout.
Values are read from the v27 checkpoint.

In [ ]:
import graphviz

d     = cfg['d_model']          # 384
heads = cfg['enc_heads']        # 8
enc_L = cfg['enc_layers']       # 8
dec_L = cfg['dec_layers']       # 2
P     = cfg.get('temporal_patch', 1)  # 3
W_p   = W // P if P > 1 else W
d_ff  = int(d * cfg['mlp_ratio'])
mr    = cfg['mask_ratio']       # 0.5
n_vis = int(N * (1 - mr))
n_params = sum(v.numel() for v in ckpt['state_dict'].values()
               if hasattr(v, 'numel')) / 1e6

g = graphviz.Digraph('v27', format='pdf',
    graph_attr={
        'rankdir': 'LR', 'bgcolor': 'white', 'fontname': 'Helvetica',
        'label': f'Weather Station Transformer (v27)  —  ≈{n_params:.1f} M parameters',
        'labelloc': 't', 'fontsize': '16', 'pad': '0.3',
        'nodesep': '0.4', 'ranksep': '0.6',
    },
    node_attr={'fontname': 'Helvetica', 'fontsize': '10', 'style': 'filled',
               'shape': 'record'},
    edge_attr={'fontname': 'Helvetica', 'fontsize': '9'},
)

# ── Input ──
g.node('input',
       label=f'{{Input X|B × {W} × {N} × {V}|12 h @ 10 min, 6 vars}}',
       fillcolor='#FDEBD0', color='#E8A838')

# ── Embeddings ──
g.node('embed',
       label=f'{{Embeddings|{{value (MLP)|position (Fourier)|topo (MLP)|'
             f'time (Fourier)|step (Fourier)}}|Σ → LayerNorm|d = {d}}}',
       fillcolor='#F5F0E0', color='#AAA')

# ── MAE Masking ──
g.node('mask',
       label=f'{{MAE Masking|whole-station|r = {mr}|{n_vis} visible / {N-n_vis} masked}}',
       fillcolor='#FFCCCC', color='#C0392B')

# ── Temporal Patching ──
if P > 1:
    g.node('patch',
           label=f'{{Temporal Patching|P = {P}|{W} steps → {W_p} patches}}',
           fillcolor='#E8EDF2', color='#8899AA')

# ── Encoder ──
enc_tokens = W_p * n_vis
g.node('encoder',
       label=f'{{Encoder (×{enc_L})|'
             f'{{Temporal self-attn|Spatial self-attn|FFN ({d}→{d_ff}→{d})}}|'
             f'h = {heads}, d = {d}|'
             f'{enc_tokens:,} tokens|'
             f'+ residual + LayerNorm|'
             f'+ stochastic depth}}',
       fillcolor='#E8EDF2', color='#8899AA')

# ── Decoder ──
g.node('decoder',
       label=f'{{Decoder (×{dec_L})|'
             f'{{Station × lead queries|N×K = {N}×{K} = {N*K:,}}}|'
             f'{{Self-attention|Cross-attention|FFN}}|'
             f'h = {heads}, d = {d}|'
             f'+ persistence base}}',
       fillcolor='#F0E8E0', color='#AA9988')

# ── Output ──
g.node('output',
       label=f'{{Outputs|B × {K} × {N} × 5|1 inpainting (Δ=0)|'
             f'+ 12 forecasts (30 min – 6 h)}}',
       fillcolor='#E8F5E9', color='#2C7A5E')

# ── Loss ──
g.node('loss',
       label='{Loss|Huber (δ=1)|uniform over K leads|all stations supervised}',
       fillcolor='#F3EFF8', color='#7B68AE')

# ── Edges ──
g.edge('input', 'embed')
g.edge('embed', 'mask')
if P > 1:
    g.edge('mask', 'patch')
    g.edge('patch', 'encoder')
else:
    g.edge('mask', 'encoder')
g.edge('encoder', 'decoder', label='  K, V  ', color='#5BA55B', fontcolor='#5BA55B')
g.edge('decoder', 'output')
g.edge('output', 'loss')

# ── Inference note ──
g.node('note',
       label='Inference: r=0 (no masking)\nall stations → encoder → decoder',
       shape='note', fillcolor='#FFFFF0', color='#CCC', fontsize='8')
g.edge('note', 'mask', style='dashed', color='#CCC', arrowhead='none')

print('Graph built')
g

In [ ]:
# Save in multiple formats
out_dir = Path('analysis_outputs/figures')
out_dir.mkdir(parents=True, exist_ok=True)

for fmt in ['pdf', 'svg', 'png']:
    g.format = fmt
    g.render(str(out_dir / 'v27_architecture_graphviz'), cleanup=True)

print(f'Saved → {out_dir}/v27_architecture_graphviz.{{pdf,svg,png}}')

---

**Which figure to use in the thesis:**

- The **matplotlib schematic** (cells above) gives pixel-level control for the main thesis figure.
- The **Graphviz schematic** is cleaner and easier to maintain — good for the thesis if the
  record-style nodes suit your layout preferences.
- The **torchview graph** is best as a supplement or appendix: it proves the architecture
  matches the description, but is too detailed for a main figure.